In [1]:
import sys
from pathlib import Path

# /app is the Docker WORKDIR; when running locally, use the repo root instead.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "utils" / "spark_session.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

GOLD_DIR = PROJECT_ROOT / "include" / "gold"
if str(GOLD_DIR) not in sys.path:
    sys.path.insert(0, str(GOLD_DIR))

# Load .env when running locally in Cursor (Docker injects env vars via docker-compose env_file).
from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / ".env")

import os

import yaml
import pyspark.sql.functions as F

from utils.spark_session import create_spark_session

with open(PROJECT_ROOT / "schema.yaml") as f:
    schema = yaml.safe_load(f)

BRONZE_PATH = schema["bronze"]["path"]
LEGAL_BRONZE = f"{BRONZE_PATH}/{schema['bronze']['tables']['legal_docs_raw']['path']}"
WIKI_BRONZE = f"{BRONZE_PATH}/{schema['bronze']['tables']['wiki_docs_raw']['path']}"

print(f"Project root: {PROJECT_ROOT}")
print(f"Legal docs path: {LEGAL_BRONZE}")
print(f"Wiki  docs path: {WIKI_BRONZE}")

# Give Spark more heap for large text tokenization in Docker.
os.environ.setdefault(
    "PYSPARK_SUBMIT_ARGS",
    "--driver-memory 4g --executor-memory 4g pyspark-shell",
)

spark = create_spark_session("explore-bronze")
print(f"Spark version: {spark.version}")

Project root: /app
Legal docs path: s3a://cs611-project/bronze/legal_docs_raw
Wiki  docs path: s3a://cs611-project/bronze/wiki_docs_raw


Picked up JAVA_TOOL_OPTIONS: --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/sun.util.calendar=ALL-UNNAMED
Picked up JAVA_TOOL_OPTIONS: --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL

:: loading settings :: url = jar:file:/usr/local/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d94bd52d-4e4a-443a-a761-3d6401f04b30;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
downloading https://repo1.maven.org/maven2/io/delta/delta-spark_2.12/3.2.1/delta-spark_2.12-3.2.1.jar ...
	[SUCCESSFUL ] io.delta#delta-spark_2.12;3.2.1!delta-spark_2.12.jar (508ms)
downloading https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-aws/3.3.4/hadoop-aws-3.3.4.jar ...
	[SUCCESSFU

Spark version: 3.5.4


In [2]:
BUCKET_ROOT = f"s3a://{schema['r2']['bucket']}"


def list_r2_tree(spark, path: str, max_depth: int = 2, max_entries: int = 30) -> None:
    """Print folders/files under an s3a:// path (R2)."""
    sc = spark.sparkContext
    URI = sc._jvm.java.net.URI
    Path = sc._jvm.org.apache.hadoop.fs.Path
    FileSystem = sc._jvm.org.apache.hadoop.fs.FileSystem

    fs = FileSystem.get(URI(path), sc._jsc.hadoopConfiguration())

    def _walk(current: str, depth: int) -> None:
        p = Path(current)
        if not fs.exists(p):
            print(f"{'  ' * depth}(not found) {current}")
            return

        statuses = list(fs.listStatus(p))
        statuses.sort(key=lambda s: (not s.isDirectory(), s.getPath().getName().lower()))

        shown = 0
        for status in statuses:
            if shown >= max_entries:
                remaining = len(statuses) - shown
                print(f"{'  ' * (depth + 1)}... and {remaining} more")
                break

            name = status.getPath().getName()
            full = status.getPath().toString()
            prefix = "  " * depth

            if status.isDirectory():
                print(f"{prefix}{name}/")
                if depth < max_depth:
                    _walk(full, depth + 1)
            else:
                size_mb = status.getLen() / (1024 * 1024)
                print(f"{prefix}{name}  ({size_mb:.2f} MB)")

            shown += 1

    print(path)
    _walk(path, 0)


# Medallion roots from schema.yaml
LAYER_PATHS = {
    "landing": schema["landing"]["path"],
    "bronze": schema["bronze"]["path"],
    "silver": schema["silver"]["path"],
    "gold": schema["gold"]["path"],
}

print(f"Bucket root: {BUCKET_ROOT}\n")
for layer, path in LAYER_PATHS.items():
    print("=" * 70)
    print(layer)
    print("=" * 70)
    list_r2_tree(spark, path, max_depth=2, max_entries=25)
    print()

# Drill into a specific table (change path / depth as needed)
# list_r2_tree(spark, f"{schema['gold']['path']}/{schema['gold']['tables']['ngram_count']['path']}", max_depth=2)

Bucket root: s3a://cs611-project

landing
s3a://cs611-project/landing
legal_docs/
  drift_data/
    EurLex_snapshot_20050101.csv  (19.79 MB)
    EurLex_snapshot_20060101.csv  (28.20 MB)
    EurLex_snapshot_20070101.csv  (24.92 MB)
    EurLex_snapshot_20080101.csv  (34.70 MB)
    EurLex_snapshot_20090101.csv  (30.85 MB)
    EurLex_snapshot_20100101.csv  (22.78 MB)
    EurLex_snapshot_20110101.csv  (29.10 MB)
    EurLex_snapshot_20120101.csv  (25.94 MB)
    EurLex_snapshot_20130101.csv  (34.31 MB)
    EurLex_snapshot_20140101.csv  (47.16 MB)
    EurLex_snapshot_20150101.csv  (35.35 MB)
    EurLex_snapshot_20160101.csv  (40.48 MB)
    EurLex_snapshot_20170101.csv  (37.47 MB)
    EurLex_snapshot_20180101.csv  (32.65 MB)
    EurLex_snapshot_20190101.csv  (24.23 MB)
  EurLex_snapshot_19870101.csv  (6.44 MB)
  EurLex_snapshot_19880101.csv  (8.06 MB)
  EurLex_snapshot_19890101.csv  (7.17 MB)
  EurLex_snapshot_19900101.csv  (8.83 MB)
  EurLex_snapshot_19910101.csv  (9.64 MB)
  EurLex_snapshot_1

## LR (embeddings + TFIDF + DCW)

In [3]:
from datetime import datetime, timezone
from typing import Any
import pickle
import tempfile
import json

TRAINING_DIR = PROJECT_ROOT / "include" / "training"
if str(TRAINING_DIR) not in sys.path:
    sys.path.insert(0, str(TRAINING_DIR))

from model_training import evaluate_multilabel

MP = schema["gold"]["model_predictions"]
PRED_BASE = f"{schema['gold']['path']}/{MP['base']}"
PREFIX = MP["file_prefix"]
SPLIT_COL = "category"
TARGET_LABELS_COL = "target_labels"
PREDICTED_LABELS_COL = "predicted_labels"

# Pin a batch date/suffix, or leave EVAL_DATE None to use the latest prediction_* on R2.
EVAL_DATE = "20260613"  # e.g. "20260613"; None = latest
EVAL_SUFFIX = "LR"  # e.g. "LR" → prediction_20260613_LR; None for RF/default batches


def _normalize_eval_suffix(suffix: str | None) -> str:
    if not suffix:
        return ""
    cleaned = suffix.strip().lstrip("_")
    return f"_{cleaned}" if cleaned else ""


def _prediction_batch_folder(eval_date: str, eval_suffix: str | None = None) -> str:
    return f"{PREFIX}{eval_date}{_normalize_eval_suffix(eval_suffix)}"

def load_bytes_from_path(path: str, spark) -> bytes:
    """Read raw bytes from s3a:// via a local temp file (reliable Py4J/Hadoop I/O)."""
    jvm = spark.sparkContext._jvm
    fd, local_path = tempfile.mkstemp()
    os.close(fd)
    try:
        hadoop_path = jvm.org.apache.hadoop.fs.Path(path)
        fs = hadoop_path.getFileSystem(spark.sparkContext._jsc.hadoopConfiguration())
        if not fs.exists(hadoop_path):
            raise FileNotFoundError(path)
        fs.copyToLocalFile(False, hadoop_path, jvm.org.apache.hadoop.fs.Path(local_path))
        with open(local_path, "rb") as handle:
            return handle.read()
    finally:
        if os.path.exists(local_path):
            os.unlink(local_path)

def load_pickle(path: str, spark) -> Any:
    return pickle.loads(load_bytes_from_path(path, spark))

def _list_prediction_child_names(spark, base: str) -> list[str]:
    sc = spark.sparkContext
    URI = sc._jvm.java.net.URI
    Path = sc._jvm.org.apache.hadoop.fs.Path
    FileSystem = sc._jvm.org.apache.hadoop.fs.FileSystem
    fs = FileSystem.get(URI(base), sc._jsc.hadoopConfiguration())
    p = Path(base)
    if not fs.exists(p):
        return []
    return [st.getPath().getName() for st in fs.listStatus(p)]


def _resolve_eval_date(spark, eval_date: str | None, eval_suffix: str | None = None) -> str:
    norm_suffix = _normalize_eval_suffix(eval_suffix)
    if eval_date:
        return eval_date
    candidates = sorted(
        name[len(PREFIX) :]
        for name in _list_prediction_child_names(spark, PRED_BASE)
        if name.startswith(PREFIX) and not name.endswith((".pkl", ".json"))
    )
    if norm_suffix:
        candidates = [c for c in candidates if c.endswith(norm_suffix)]
    else:
        candidates = [c for c in candidates if c[:8].isdigit() and len(c) == 8]
    if not candidates:
        raise FileNotFoundError(
            f"No prediction batches under {PRED_BASE}. "
            "Run training with --predict-stage predict, then metrics."
        )
    return candidates[-1]

def _label_universe_from_predictions(predictions) -> list[str]:
    labels = (
        predictions.select(F.explode(F.col(TARGET_LABELS_COL)).alias("lbl"))
        .union(predictions.select(F.explode(F.col(PREDICTED_LABELS_COL)).alias("lbl")))
        .select("lbl")
        .distinct()
        .orderBy("lbl")
    )
    return [row.lbl for row in labels.collect()]


def load_prediction_metrics(
    spark, eval_date: str | None = None, eval_suffix: str | None = None
) -> tuple[dict, str]:
    """Load metrics from manifest if present; otherwise compute from prediction Delta."""
    batch_date = _resolve_eval_date(spark, eval_date, eval_suffix)
    batch_folder = _prediction_batch_folder(batch_date, eval_suffix)
    for ext, loader in (
        (".pkl", lambda path: load_pickle(path, spark)),
        (".json", lambda path: json.loads(load_bytes_from_path(path, spark).decode("utf-8"))),
    ):
        path = f"{PRED_BASE}/{batch_folder}{ext}"
        try:
            manifest = loader(path)
            return manifest["metrics"], path
        except FileNotFoundError:
            continue

    delta_path = f"{PRED_BASE}/{batch_folder}"
    delta_names = _list_prediction_child_names(spark, PRED_BASE)
    if batch_folder not in delta_names:
        raise FileNotFoundError(f"No prediction outputs for batch {batch_folder} under {PRED_BASE}")

    print(f"No manifest found; computing metrics from Delta: {delta_path}")
    predictions = spark.read.format("delta").load(delta_path)
    label_list = _label_universe_from_predictions(predictions)
    print(f"Label universe from predictions: {len(label_list)} labels")
    metrics = evaluate_multilabel(predictions, label_list)
    return metrics, delta_path


metrics, source_path = load_prediction_metrics(spark, EVAL_DATE, EVAL_SUFFIX)
print(f"Metrics source: {source_path}")

for split in ("holdout_val", "holdout_test", "holdout_oot", "holdout_overall"):
    m = metrics[split]
    print(f"\n=== {split} ===")
    print(f"  documents:      {m['documents']}")
    print(f"  micro_f1:        {m['micro_f1']:.4f}")
    print(f"  macro_f1:        {m['macro_f1']:.4f}")
    print(f"  exact_match:     {m['exact_match_ratio']:.4f}")
    print(f"  hamming_loss:    {m['hamming_loss']:.4f}")
    print(f"  micro_p / micro_r: {m['micro_precision']:.4f} / {m['micro_recall']:.4f}")

No manifest found; computing metrics from Delta: s3a://cs611-project/gold/model_predictions/prediction_20260613_LR


Label universe from predictions: 15 labels


[Stage 356:>                                                        (0 + 1) / 1]

Metrics source: s3a://cs611-project/gold/model_predictions/prediction_20260613_LR

=== holdout_val ===
  documents:      5338
  micro_f1:        0.8313
  macro_f1:        0.7154
  exact_match:     0.4565
  hamming_loss:    0.0940
  micro_p / micro_r: 0.7955 / 0.8704

=== holdout_test ===
  documents:      2668
  micro_f1:        0.8375
  macro_f1:        0.7247
  exact_match:     0.4640
  hamming_loss:    0.0903
  micro_p / micro_r: 0.8043 / 0.8735

=== holdout_oot ===
  documents:      1687
  micro_f1:        0.7657
  macro_f1:        0.6674
  exact_match:     0.2928
  hamming_loss:    0.1358
  micro_p / micro_r: 0.7056 / 0.8369

=== holdout_overall ===
  documents:      9693
  micro_f1:        0.8212
  macro_f1:        0.7098
  exact_match:     0.4301
  hamming_loss:    0.1003
  micro_p / micro_r: 0.7812 / 0.8654


In [4]:
# Threshold sweep from saved prob_* columns (no Docker re-predict needed).
# Re-run --predict-stage predict once after rebuilding the image to populate probabilities.

from model_training import apply_multilabel_threshold, label_prob_column_map, prob_columns_in_df

delta_path = f"{PRED_BASE}/{_prediction_batch_folder(EVAL_DATE, EVAL_SUFFIX)}"
predictions = spark.read.format("delta").load(delta_path)
prob_cols = prob_columns_in_df(predictions)

if not prob_cols:
    print(
        "This prediction batch has no prob_* columns yet.\n"
        "Rebuild the image, then re-run:\n"
        "  --predict-only --predict-stage predict --prediction-date",
        EVAL_DATE,
    )
else:
    print(f"Loaded {len(prob_cols)} probability columns from {delta_path}")
    label_list = _label_universe_from_predictions(predictions)
    prob_map = {label: col for label, col in label_prob_column_map(label_list).items() if col in prob_cols}

    print(f"\n{'threshold':>10}  {'micro_f1':>9}  {'accuracy':>9}  {'micro_r':>9}  {'micro_p':>9}")
    print("-" * 52)
    for threshold in (0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80):
        scored = apply_multilabel_threshold(predictions, label_list, threshold, prob_column_map=prob_map)
        overall = evaluate_multilabel(scored, label_list)["holdout_overall"]
        accuracy = overall.get("accuracy", overall.get("exact_match_ratio"))
        print(
            f"{threshold:10.2f}  {overall['micro_f1']:9.4f}  {accuracy:9.4f}  "
            f"{overall['micro_recall']:9.4f}  {overall['micro_precision']:9.4f}"
        )

Loaded 15 probability columns from s3a://cs611-project/gold/model_predictions/prediction_20260613_LR



 threshold   micro_f1   accuracy    micro_r    micro_p
----------------------------------------------------


      0.50     0.8212     0.4301     0.8654     0.7812


      0.55     0.8212     0.4305     0.8648     0.7817


      0.60     0.8211     0.4305     0.8641     0.7822


      0.65     0.8209     0.4303     0.8633     0.7825


      0.70     0.8210     0.4303     0.8627     0.7831


      0.75     0.8208     0.4304     0.8616     0.7837


[Stage 2748:>                                                       (0 + 1) / 1]

      0.80     0.8205     0.4296     0.8604     0.7841


## RF (embeddings + TFIDF + DCW)

In [7]:
from datetime import datetime, timezone
from typing import Any
import pickle
import tempfile
import json

TRAINING_DIR = PROJECT_ROOT / "include" / "training"
if str(TRAINING_DIR) not in sys.path:
    sys.path.insert(0, str(TRAINING_DIR))

from model_training import evaluate_multilabel

MP = schema["gold"]["model_predictions"]
PRED_BASE = f"{schema['gold']['path']}/{MP['base']}"
PREFIX = MP["file_prefix"]
SPLIT_COL = "category"
TARGET_LABELS_COL = "target_labels"
PREDICTED_LABELS_COL = "predicted_labels"

# Pin a batch date/suffix, or leave EVAL_DATE None to use the latest prediction_* on R2.
EVAL_DATE = "20260612"  # e.g. "20260613"; None = latest
EVAL_SUFFIX = "RF"  # e.g. "LR" → prediction_20260613_LR; None for RF/default batches


def _normalize_eval_suffix(suffix: str | None) -> str:
    if not suffix:
        return ""
    cleaned = suffix.strip().lstrip("_")
    return f"_{cleaned}" if cleaned else ""


def _prediction_batch_folder(eval_date: str, eval_suffix: str | None = None) -> str:
    return f"{PREFIX}{eval_date}{_normalize_eval_suffix(eval_suffix)}"

def load_bytes_from_path(path: str, spark) -> bytes:
    """Read raw bytes from s3a:// via a local temp file (reliable Py4J/Hadoop I/O)."""
    jvm = spark.sparkContext._jvm
    fd, local_path = tempfile.mkstemp()
    os.close(fd)
    try:
        hadoop_path = jvm.org.apache.hadoop.fs.Path(path)
        fs = hadoop_path.getFileSystem(spark.sparkContext._jsc.hadoopConfiguration())
        if not fs.exists(hadoop_path):
            raise FileNotFoundError(path)
        fs.copyToLocalFile(False, hadoop_path, jvm.org.apache.hadoop.fs.Path(local_path))
        with open(local_path, "rb") as handle:
            return handle.read()
    finally:
        if os.path.exists(local_path):
            os.unlink(local_path)

def load_pickle(path: str, spark) -> Any:
    return pickle.loads(load_bytes_from_path(path, spark))

def _list_prediction_child_names(spark, base: str) -> list[str]:
    sc = spark.sparkContext
    URI = sc._jvm.java.net.URI
    Path = sc._jvm.org.apache.hadoop.fs.Path
    FileSystem = sc._jvm.org.apache.hadoop.fs.FileSystem
    fs = FileSystem.get(URI(base), sc._jsc.hadoopConfiguration())
    p = Path(base)
    if not fs.exists(p):
        return []
    return [st.getPath().getName() for st in fs.listStatus(p)]


def _resolve_eval_date(spark, eval_date: str | None, eval_suffix: str | None = None) -> str:
    norm_suffix = _normalize_eval_suffix(eval_suffix)
    if eval_date:
        return eval_date
    candidates = sorted(
        name[len(PREFIX) :]
        for name in _list_prediction_child_names(spark, PRED_BASE)
        if name.startswith(PREFIX) and not name.endswith((".pkl", ".json"))
    )
    if norm_suffix:
        candidates = [c for c in candidates if c.endswith(norm_suffix)]
    else:
        candidates = [c for c in candidates if c[:8].isdigit() and len(c) == 8]
    if not candidates:
        raise FileNotFoundError(
            f"No prediction batches under {PRED_BASE}. "
            "Run training with --predict-stage predict, then metrics."
        )
    return candidates[-1]

def _label_universe_from_predictions(predictions) -> list[str]:
    labels = (
        predictions.select(F.explode(F.col(TARGET_LABELS_COL)).alias("lbl"))
        .union(predictions.select(F.explode(F.col(PREDICTED_LABELS_COL)).alias("lbl")))
        .select("lbl")
        .distinct()
        .orderBy("lbl")
    )
    return [row.lbl for row in labels.collect()]


def load_prediction_metrics(
    spark, eval_date: str | None = None, eval_suffix: str | None = None
) -> tuple[dict, str]:
    """Load metrics from manifest if present; otherwise compute from prediction Delta."""
    batch_date = _resolve_eval_date(spark, eval_date, eval_suffix)
    batch_folder = _prediction_batch_folder(batch_date, eval_suffix)
    for ext, loader in (
        (".pkl", lambda path: load_pickle(path, spark)),
        (".json", lambda path: json.loads(load_bytes_from_path(path, spark).decode("utf-8"))),
    ):
        path = f"{PRED_BASE}/{batch_folder}{ext}"
        try:
            manifest = loader(path)
            return manifest["metrics"], path
        except FileNotFoundError:
            continue

    delta_path = f"{PRED_BASE}/{batch_folder}"
    delta_names = _list_prediction_child_names(spark, PRED_BASE)
    if batch_folder not in delta_names:
        raise FileNotFoundError(f"No prediction outputs for batch {batch_folder} under {PRED_BASE}")

    print(f"No manifest found; computing metrics from Delta: {delta_path}")
    predictions = spark.read.format("delta").load(delta_path)
    label_list = _label_universe_from_predictions(predictions)
    print(f"Label universe from predictions: {len(label_list)} labels")
    metrics = evaluate_multilabel(predictions, label_list)
    return metrics, delta_path


metrics, source_path = load_prediction_metrics(spark, EVAL_DATE, EVAL_SUFFIX)
print(f"Metrics source: {source_path}")

for split in ("holdout_val", "holdout_test", "holdout_oot", "holdout_overall"):
    m = metrics[split]
    print(f"\n=== {split} ===")
    print(f"  documents:      {m['documents']}")
    print(f"  micro_f1:        {m['micro_f1']:.4f}")
    print(f"  macro_f1:        {m['macro_f1']:.4f}")
    print(f"  exact_match:     {m['exact_match_ratio']:.4f}")
    print(f"  hamming_loss:    {m['hamming_loss']:.4f}")
    print(f"  micro_p / micro_r: {m['micro_precision']:.4f} / {m['micro_recall']:.4f}")

No manifest found; computing metrics from Delta: s3a://cs611-project/gold/model_predictions/prediction_20260612_RF


Label universe from predictions: 15 labels


[Stage 3107:>                                                       (0 + 1) / 1]

Metrics source: s3a://cs611-project/gold/model_predictions/prediction_20260612_RF

=== holdout_val ===
  documents:      5338
  micro_f1:        0.7825
  macro_f1:        0.5662
  exact_match:     0.4262
  hamming_loss:    0.1064
  micro_p / micro_r: 0.8580 / 0.7193

=== holdout_test ===
  documents:      2668
  micro_f1:        0.7850
  macro_f1:        0.5664
  exact_match:     0.4250
  hamming_loss:    0.1054
  micro_p / micro_r: 0.8602 / 0.7218

=== holdout_oot ===
  documents:      1687
  micro_f1:        0.7436
  macro_f1:        0.5098
  exact_match:     0.3871
  hamming_loss:    0.1252
  micro_p / micro_r: 0.8131 / 0.6850

=== holdout_overall ===
  documents:      9693
  micro_f1:        0.7764
  macro_f1:        0.5559
  exact_match:     0.4191
  hamming_loss:    0.1094
  micro_p / micro_r: 0.8508 / 0.7140


In [8]:
# Threshold sweep from saved prob_* columns (no Docker re-predict needed).
# Re-run --predict-stage predict once after rebuilding the image to populate probabilities.

from model_training import apply_multilabel_threshold, label_prob_column_map, prob_columns_in_df

delta_path = f"{PRED_BASE}/{_prediction_batch_folder(EVAL_DATE, EVAL_SUFFIX)}"
predictions = spark.read.format("delta").load(delta_path)
prob_cols = prob_columns_in_df(predictions)

if not prob_cols:
    print(
        "This prediction batch has no prob_* columns yet.\n"
        "Rebuild the image, then re-run:\n"
        "  --predict-only --predict-stage predict --prediction-date",
        EVAL_DATE,
    )
else:
    print(f"Loaded {len(prob_cols)} probability columns from {delta_path}")
    label_list = _label_universe_from_predictions(predictions)
    prob_map = {label: col for label, col in label_prob_column_map(label_list).items() if col in prob_cols}

    print(f"\n{'threshold':>10}  {'micro_f1':>9}  {'accuracy':>9}  {'micro_r':>9}  {'micro_p':>9}")
    print("-" * 52)
    for threshold in (0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80):
        scored = apply_multilabel_threshold(predictions, label_list, threshold, prob_column_map=prob_map)
        overall = evaluate_multilabel(scored, label_list)["holdout_overall"]
        accuracy = overall.get("accuracy", overall.get("exact_match_ratio"))
        print(
            f"{threshold:10.2f}  {overall['micro_f1']:9.4f}  {accuracy:9.4f}  "
            f"{overall['micro_recall']:9.4f}  {overall['micro_precision']:9.4f}"
        )

Loaded 15 probability columns from s3a://cs611-project/gold/model_predictions/prediction_20260612_RF



 threshold   micro_f1   accuracy    micro_r    micro_p
----------------------------------------------------


      0.50     0.7764     0.4191     0.7140     0.8508


      0.55     0.7647     0.4009     0.6849     0.8655


      0.60     0.7516     0.3808     0.6540     0.8833


      0.65     0.7335     0.3550     0.6166     0.9050


      0.70     0.7085     0.3192     0.5719     0.9308


      0.75     0.6643     0.2528     0.5087     0.9569


[Stage 5499:>                                                       (0 + 1) / 1]

      0.80     0.5847     0.1512     0.4168     0.9796


In [9]:
# Threshold sweep from saved prob_* columns (no Docker re-predict needed).
# Re-run --predict-stage predict once after rebuilding the image to populate probabilities.

from model_training import apply_multilabel_threshold, label_prob_column_map, prob_columns_in_df

delta_path = f"{PRED_BASE}/{_prediction_batch_folder(EVAL_DATE, EVAL_SUFFIX)}"
predictions = spark.read.format("delta").load(delta_path)
prob_cols = prob_columns_in_df(predictions)

if not prob_cols:
    print(
        "This prediction batch has no prob_* columns yet.\n"
        "Rebuild the image, then re-run:\n"
        "  --predict-only --predict-stage predict --prediction-date",
        EVAL_DATE,
    )
else:
    print(f"Loaded {len(prob_cols)} probability columns from {delta_path}")
    label_list = _label_universe_from_predictions(predictions)
    prob_map = {label: col for label, col in label_prob_column_map(label_list).items() if col in prob_cols}

    print(f"\n{'threshold':>10}  {'micro_f1':>9}  {'accuracy':>9}  {'micro_r':>9}  {'micro_p':>9}")
    print("-" * 52)
    for threshold in (0.30, 0.35, 0.40, 0.45, 0.50):
        scored = apply_multilabel_threshold(predictions, label_list, threshold, prob_column_map=prob_map)
        overall = evaluate_multilabel(scored, label_list)["holdout_overall"]
        accuracy = overall.get("accuracy", overall.get("exact_match_ratio"))
        print(
            f"{threshold:10.2f}  {overall['micro_f1']:9.4f}  {accuracy:9.4f}  "
            f"{overall['micro_recall']:9.4f}  {overall['micro_precision']:9.4f}"
        )

Loaded 15 probability columns from s3a://cs611-project/gold/model_predictions/prediction_20260612_RF



 threshold   micro_f1   accuracy    micro_r    micro_p
----------------------------------------------------


      0.30     0.7976     0.3927     0.8242     0.7726


      0.35     0.7961     0.4143     0.7947     0.7975


      0.40     0.7903     0.4236     0.7665     0.8155


      0.45     0.7852     0.4264     0.7410     0.8350


[Stage 7211:>                                                       (0 + 1) / 1]

      0.50     0.7764     0.4191     0.7140     0.8508


## LR (TFIDF + DCW)

In [6]:
from datetime import datetime, timezone
from typing import Any
import pickle
import tempfile
import json

TRAINING_DIR = PROJECT_ROOT / "include" / "training"
if str(TRAINING_DIR) not in sys.path:
    sys.path.insert(0, str(TRAINING_DIR))

from model_training import evaluate_multilabel

MP = schema["gold"]["model_predictions"]
PRED_BASE = f"{schema['gold']['path']}/{MP['base']}"
PREFIX = MP["file_prefix"]
SPLIT_COL = "category"
TARGET_LABELS_COL = "target_labels"
PREDICTED_LABELS_COL = "predicted_labels"

# Pin a batch date/suffix, or leave EVAL_DATE None to use the latest prediction_* on R2.
EVAL_DATE = "20260613"  # e.g. "20260613"; None = latest
EVAL_SUFFIX = "tfidf_dcw"  # e.g. "LR" → prediction_20260613_LR; None for RF/default batches


def _normalize_eval_suffix(suffix: str | None) -> str:
    if not suffix:
        return ""
    cleaned = suffix.strip().lstrip("_")
    return f"_{cleaned}" if cleaned else ""


def _prediction_batch_folder(eval_date: str, eval_suffix: str | None = None) -> str:
    return f"{PREFIX}{eval_date}{_normalize_eval_suffix(eval_suffix)}"

def load_bytes_from_path(path: str, spark) -> bytes:
    """Read raw bytes from s3a:// via a local temp file (reliable Py4J/Hadoop I/O)."""
    jvm = spark.sparkContext._jvm
    fd, local_path = tempfile.mkstemp()
    os.close(fd)
    try:
        hadoop_path = jvm.org.apache.hadoop.fs.Path(path)
        fs = hadoop_path.getFileSystem(spark.sparkContext._jsc.hadoopConfiguration())
        if not fs.exists(hadoop_path):
            raise FileNotFoundError(path)
        fs.copyToLocalFile(False, hadoop_path, jvm.org.apache.hadoop.fs.Path(local_path))
        with open(local_path, "rb") as handle:
            return handle.read()
    finally:
        if os.path.exists(local_path):
            os.unlink(local_path)

def load_pickle(path: str, spark) -> Any:
    return pickle.loads(load_bytes_from_path(path, spark))

def _list_prediction_child_names(spark, base: str) -> list[str]:
    sc = spark.sparkContext
    URI = sc._jvm.java.net.URI
    Path = sc._jvm.org.apache.hadoop.fs.Path
    FileSystem = sc._jvm.org.apache.hadoop.fs.FileSystem
    fs = FileSystem.get(URI(base), sc._jsc.hadoopConfiguration())
    p = Path(base)
    if not fs.exists(p):
        return []
    return [st.getPath().getName() for st in fs.listStatus(p)]


def _resolve_eval_date(spark, eval_date: str | None, eval_suffix: str | None = None) -> str:
    norm_suffix = _normalize_eval_suffix(eval_suffix)
    if eval_date:
        return eval_date
    candidates = sorted(
        name[len(PREFIX) :]
        for name in _list_prediction_child_names(spark, PRED_BASE)
        if name.startswith(PREFIX) and not name.endswith((".pkl", ".json"))
    )
    if norm_suffix:
        candidates = [c for c in candidates if c.endswith(norm_suffix)]
    else:
        candidates = [c for c in candidates if c[:8].isdigit() and len(c) == 8]
    if not candidates:
        raise FileNotFoundError(
            f"No prediction batches under {PRED_BASE}. "
            "Run training with --predict-stage predict, then metrics."
        )
    return candidates[-1]

def _label_universe_from_predictions(predictions) -> list[str]:
    labels = (
        predictions.select(F.explode(F.col(TARGET_LABELS_COL)).alias("lbl"))
        .union(predictions.select(F.explode(F.col(PREDICTED_LABELS_COL)).alias("lbl")))
        .select("lbl")
        .distinct()
        .orderBy("lbl")
    )
    return [row.lbl for row in labels.collect()]


def load_prediction_metrics(
    spark, eval_date: str | None = None, eval_suffix: str | None = None
) -> tuple[dict, str]:
    """Load metrics from manifest if present; otherwise compute from prediction Delta."""
    batch_date = _resolve_eval_date(spark, eval_date, eval_suffix)
    batch_folder = _prediction_batch_folder(batch_date, eval_suffix)
    for ext, loader in (
        (".pkl", lambda path: load_pickle(path, spark)),
        (".json", lambda path: json.loads(load_bytes_from_path(path, spark).decode("utf-8"))),
    ):
        path = f"{PRED_BASE}/{batch_folder}{ext}"
        try:
            manifest = loader(path)
            return manifest["metrics"], path
        except FileNotFoundError:
            continue

    delta_path = f"{PRED_BASE}/{batch_folder}"
    delta_names = _list_prediction_child_names(spark, PRED_BASE)
    if batch_folder not in delta_names:
        raise FileNotFoundError(f"No prediction outputs for batch {batch_folder} under {PRED_BASE}")

    print(f"No manifest found; computing metrics from Delta: {delta_path}")
    predictions = spark.read.format("delta").load(delta_path)
    label_list = _label_universe_from_predictions(predictions)
    print(f"Label universe from predictions: {len(label_list)} labels")
    metrics = evaluate_multilabel(predictions, label_list)
    return metrics, delta_path


metrics, source_path = load_prediction_metrics(spark, EVAL_DATE, EVAL_SUFFIX)
print(f"Metrics source: {source_path}")

for split in ("holdout_val", "holdout_test", "holdout_oot", "holdout_overall"):
    m = metrics[split]
    print(f"\n=== {split} ===")
    print(f"  documents:      {m['documents']}")
    print(f"  micro_f1:        {m['micro_f1']:.4f}")
    print(f"  macro_f1:        {m['macro_f1']:.4f}")
    print(f"  exact_match:     {m['exact_match_ratio']:.4f}")
    print(f"  hamming_loss:    {m['hamming_loss']:.4f}")
    print(f"  micro_p / micro_r: {m['micro_precision']:.4f} / {m['micro_recall']:.4f}")

No manifest found; computing metrics from Delta: s3a://cs611-project/gold/model_predictions/prediction_20260613_tfidf_dcw


Label universe from predictions: 15 labels


[Stage 356:>                                                        (0 + 1) / 1]

Metrics source: s3a://cs611-project/gold/model_predictions/prediction_20260613_tfidf_dcw

=== holdout_val ===
  documents:      5338
  micro_f1:        0.8312
  macro_f1:        0.7151
  exact_match:     0.4571
  hamming_loss:    0.0937
  micro_p / micro_r: 0.7983 / 0.8670

=== holdout_test ===
  documents:      2668
  micro_f1:        0.8363
  macro_f1:        0.7226
  exact_match:     0.4693
  hamming_loss:    0.0907
  micro_p / micro_r: 0.8052 / 0.8699

=== holdout_oot ===
  documents:      1687
  micro_f1:        0.7704
  macro_f1:        0.6730
  exact_match:     0.3118
  hamming_loss:    0.1314
  micro_p / micro_r: 0.7176 / 0.8317

=== holdout_overall ===
  documents:      9693
  micro_f1:        0.8218
  macro_f1:        0.7105
  exact_match:     0.4352
  hamming_loss:    0.0994
  micro_p / micro_r: 0.7854 / 0.8617


In [7]:
# Threshold sweep from saved prob_* columns (no Docker re-predict needed).
# Re-run --predict-stage predict once after rebuilding the image to populate probabilities.

from model_training import apply_multilabel_threshold, label_prob_column_map, prob_columns_in_df

delta_path = f"{PRED_BASE}/{_prediction_batch_folder(EVAL_DATE, EVAL_SUFFIX)}"
predictions = spark.read.format("delta").load(delta_path)
prob_cols = prob_columns_in_df(predictions)

if not prob_cols:
    print(
        "This prediction batch has no prob_* columns yet.\n"
        "Rebuild the image, then re-run:\n"
        "  --predict-only --predict-stage predict --prediction-date",
        EVAL_DATE,
    )
else:
    print(f"Loaded {len(prob_cols)} probability columns from {delta_path}")
    label_list = _label_universe_from_predictions(predictions)
    prob_map = {label: col for label, col in label_prob_column_map(label_list).items() if col in prob_cols}

    print(f"\n{'threshold':>10}  {'micro_f1':>9}  {'accuracy':>9}  {'micro_r':>9}  {'micro_p':>9}")
    print("-" * 52)
    for threshold in (0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80):
        scored = apply_multilabel_threshold(predictions, label_list, threshold, prob_column_map=prob_map)
        overall = evaluate_multilabel(scored, label_list)["holdout_overall"]
        accuracy = overall.get("accuracy", overall.get("exact_match_ratio"))
        print(
            f"{threshold:10.2f}  {overall['micro_f1']:9.4f}  {accuracy:9.4f}  "
            f"{overall['micro_recall']:9.4f}  {overall['micro_precision']:9.4f}"
        )

Loaded 15 probability columns from s3a://cs611-project/gold/model_predictions/prediction_20260613_tfidf_dcw



 threshold   micro_f1   accuracy    micro_r    micro_p
----------------------------------------------------


      0.50     0.8218     0.4352     0.8617     0.7854


      0.55     0.8217     0.4352     0.8611     0.7857


      0.60     0.8214     0.4345     0.8602     0.7861


      0.65     0.8213     0.4344     0.8593     0.7865


      0.70     0.8213     0.4344     0.8587     0.7871


      0.75     0.8213     0.4346     0.8578     0.7878


[Stage 2748:>                                                       (0 + 1) / 1]

      0.80     0.8212     0.4343     0.8566     0.7885


In [8]:
# Threshold sweep from saved prob_* columns (no Docker re-predict needed).
# Re-run --predict-stage predict once after rebuilding the image to populate probabilities.

from model_training import apply_multilabel_threshold, label_prob_column_map, prob_columns_in_df

delta_path = f"{PRED_BASE}/{_prediction_batch_folder(EVAL_DATE, EVAL_SUFFIX)}"
predictions = spark.read.format("delta").load(delta_path)
prob_cols = prob_columns_in_df(predictions)

if not prob_cols:
    print(
        "This prediction batch has no prob_* columns yet.\n"
        "Rebuild the image, then re-run:\n"
        "  --predict-only --predict-stage predict --prediction-date",
        EVAL_DATE,
    )
else:
    print(f"Loaded {len(prob_cols)} probability columns from {delta_path}")
    label_list = _label_universe_from_predictions(predictions)
    prob_map = {label: col for label, col in label_prob_column_map(label_list).items() if col in prob_cols}

    print(f"\n{'threshold':>10}  {'micro_f1':>9}  {'accuracy':>9}  {'micro_r':>9}  {'micro_p':>9}")
    print("-" * 52)
    for threshold in (0.30, 0.35, 0.40, 0.45, 0.50):
        scored = apply_multilabel_threshold(predictions, label_list, threshold, prob_column_map=prob_map)
        overall = evaluate_multilabel(scored, label_list)["holdout_overall"]
        accuracy = overall.get("accuracy", overall.get("exact_match_ratio"))
        print(
            f"{threshold:10.2f}  {overall['micro_f1']:9.4f}  {accuracy:9.4f}  "
            f"{overall['micro_recall']:9.4f}  {overall['micro_precision']:9.4f}"
        )

Loaded 15 probability columns from s3a://cs611-project/gold/model_predictions/prediction_20260613_tfidf_dcw



 threshold   micro_f1   accuracy    micro_r    micro_p
----------------------------------------------------


      0.30     0.8215     0.4337     0.8640     0.7829


      0.35     0.8216     0.4343     0.8635     0.7836


      0.40     0.8217     0.4348     0.8629     0.7843


      0.45     0.8217     0.4346     0.8622     0.7848


[Stage 4460:>                                                       (0 + 1) / 1]

      0.50     0.8218     0.4352     0.8617     0.7854
